# 9. The weather at the sites

Every result so far took out the weather of each site's soum, averaged over the soum's summer grazing range. A soum can be 100 km across, and the design names this as a risk: rain on the soum's range is not rain at the site. This notebook takes the weather at each site's own place and repeats the trends.

The weather comes from ERA5-Land, a reconstruction on a grid of about 11 km, at the grid cell that holds each plot. docs/design.md set the rule before the extraction: the same seasons as the release's soum table, with rain built two ways, as all precipitation and as precipitation less snowfall, and the way that agrees better with the soum table is used.

- **Reads** `sites.csv`, `site_year.csv`, `flags.csv`, `modis_jul_aug_median.csv` and `landsat_site_years.csv` from `data/processed/`, the weather table of the release, and ERA5-Land from Earth Engine on the first run.
- **Writes** `era5_land_monthly.csv` to `data/processed/`.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mn_desertification import data, paths, quality, trends
from mn_desertification import satellite as sat
from mn_desertification.rules import FLOOR, LONG_MIN_YEARS, MAIN_YEARS, MIN_YEARS, ZONES
from mn_desertification.weather import fit_weather_model, season_weather, weather_remainders

site_info = pd.read_csv(paths.PROCESSED_DIR / "sites.csv")
soum_weather = data.read_soum_weather()

# ERA5-Land at each site, March to August of 1989 to 2024. About a minute the first time, then read from disk.
monthly_file = paths.PROCESSED_DIR / "era5_land_monthly.csv"
if monthly_file.exists():
    monthly = pd.read_csv(monthly_file)
else:
    sat.connect()
    monthly = sat.monthly_weather_at_sites(site_info, range(1989, 2025))
    monthly.to_csv(monthly_file, index=False)
print("Rows:", len(monthly), "| sites:", monthly["gid"].nunique(), "| years:", monthly["year"].min(), "to", monthly["year"].max(),
      "| missing values:", int(monthly.isna().sum().sum()))
monthly.groupby("month")[["precipitation_mm", "snowfall_mm", "temperature_c"]].mean().round(1).T

Rows: 321408 | sites: 1488 | years: 1989 to 2024 | missing values: 0


month,3,4,5,6,7,8
precipitation_mm,9.8,18.4,29.8,49.7,69.4,58.7
snowfall_mm,9.0,12.4,8.8,1.3,0.1,0.6
temperature_c,-5.6,2.5,9.5,15.6,17.9,15.8


Much of spring's precipitation falls as snow: nearly all of it in March, and still more than a quarter in May.

## Which rain does the release's table hold?

The sites of each soum are averaged and set beside the release's soum table.

In [2]:
# Which rain does the release's soum table hold? The sites of each soum are averaged and set beside it
match_rows = []
for rain in ["all", "less snow"]:
    seasons = season_weather(monthly, rain).merge(site_info[["gid", "asid"]], on="gid")
    by_soum = seasons.groupby(["asid", "year"])[["rain_spring", "rain_summer", "temp_summer"]].mean().reset_index()
    both = by_soum.merge(soum_weather, on=["asid", "year"], suffixes=("", "_soum"))
    for name in ["rain_spring", "rain_summer", "temp_summer"]:
        match_rows.append({"rain": rain, "variable": name, "soum-years": len(both),
                           "correlation": both[name].corr(both[name + "_soum"]),
                           "mean at the sites": both[name].mean(), "mean in the soum table": both[name + "_soum"].mean()})
pd.DataFrame(match_rows).round({"correlation": 3, "mean at the sites": 1, "mean in the soum table": 1})

,rain,variable,soum-years,correlation,mean at the sites,mean in the soum table
0,all,rain_spring,11844,0.966,57.4,56.7
1,all,rain_summer,11844,0.952,177.7,176.4
2,all,temp_summer,11844,0.940,16.7,16.7
3,less snow,rain_spring,11844,0.685,28.0,56.7
4,less snow,rain_summer,11844,0.951,175.9,176.4
5,less snow,temp_summer,11844,0.940,16.7,16.7


**The release's rain is all precipitation, snow included.** With snow, the sites' spring rain matches the soum table closely. With snowfall taken out it is half as large and matches poorly. Summer rain and summer temperature match either way. By the rule, all precipitation is used, and what the earlier notebooks call rain includes the snow of March to May.

## How far is a site's weather from its soum's?

In [3]:
# The weather at each site, with all precipitation as rain, beside the weather of its soum
site_weather = season_weather(monthly, "all")
site_weather = site_weather.merge(site_info[["gid", "asid", "zone"]], on="gid")
beside = site_weather.merge(soum_weather, on=["asid", "year"], suffixes=("", "_soum"))
decade = beside[beside["year"].between(*MAIN_YEARS)]

apart = decade["rain_growing"] / decade["rain_growing_soum"] - 1
print("Site-years 2011-2020:", len(decade))
print("Growing-season rain, site against soum: correlation", round(decade["rain_growing"].corr(decade["rain_growing_soum"]), 3),
      "| typical difference", f"{apart.abs().median():.0%}", "| more than 20% apart:", f"{(apart.abs() > 0.2).mean():.0%}")
print("Summer temperature, site against soum: correlation", round(decade["temp_summer"].corr(decade["temp_summer_soum"]), 3),
      "| typical difference", round((decade["temp_summer"] - decade["temp_summer_soum"]).abs().median(), 2), "degrees")

# The same after each site's own average is taken out, which is what the weather model uses
within = decade[["rain_growing", "rain_growing_soum", "temp_summer", "temp_summer_soum"]].copy()
within[["rain_growing", "rain_growing_soum"]] = np.log10(within[["rain_growing", "rain_growing_soum"]])
within = within - within.groupby(decade["gid"]).transform("mean")
print("Year-to-year changes, site against soum: rain", round(within["rain_growing"].corr(within["rain_growing_soum"]), 3),
      "| temperature", round(within["temp_summer"].corr(within["temp_summer_soum"]), 3))

Site-years 2011-2020: 14880
Growing-season rain, site against soum: correlation 0.91 | typical difference 8% | more than 20% apart: 19%
Summer temperature, site against soum: correlation 0.889 | typical difference 0.61 degrees
Year-to-year changes, site against soum: rain 0.959 | temperature 0.991


- **Levels differ.** A typical site is 8% away from its soum in growing-season rain, and a fifth of the site-years are more than 20% away.
- **Changes from year to year hardly differ.** A wet year in the soum is a wet year at the site. The weather model works with those changes, each site against its own average, so little room is left for the results to move.

## The weather model and the trends, with each site's own weather

In [4]:
# Does a site's own weather explain its biomass better than its soum's weather?
flags = pd.read_csv(paths.PROCESSED_DIR / "flags.csv")
site_year = pd.read_csv(paths.PROCESSED_DIR / "site_year.csv")
biomass = quality.drop_flagged(site_year[site_year["year"].between(*MAIN_YEARS)], flags)[["gid", "year", "bms"]].copy()
biomass["bms_floored"] = biomass["bms"].clip(lower=FLOOR)

def with_site_weather(values):
    return values.merge(site_weather, on=["gid", "year"], how="left")

def with_soum_weather(values):
    return values.merge(site_info[["gid", "asid"]], on="gid").merge(soum_weather, on=["asid", "year"], how="left")

fits = {"soum": fit_weather_model(with_soum_weather(biomass), "bms_floored").effects.set_index("zone"),
        "site": fit_weather_model(with_site_weather(biomass), "bms_floored").effects.set_index("zone")}
pd.concat(fits, axis=1).reindex(ZONES).round(3)

soum                            site                \
                        b_rain c_temp     R2 site-years b_rain c_temp     R2   
zone                                                                           
Desert zone              1.040 -0.109  0.148        951  0.936 -0.111  0.153   
Semi desert steppe zone  0.834 -0.153  0.150       2810  0.790 -0.153  0.150   
Steppe zone              0.765 -0.109  0.139       5763  0.666 -0.113  0.135   
Forest steppe zone       0.697 -0.078  0.075       3402  0.602 -0.084  0.074   
Mountain taiga belt      0.986 -0.098  0.094        441  1.000 -0.101  0.099   

                                    
                        site-years  
zone                                
Desert zone                    951  
Semi desert steppe zone       2810  
Steppe zone                   5763  
Forest steppe zone            3402  
Mountain taiga belt            441

In [5]:
def trend_row(name, table, value, period, min_years):
    """The trend of the yearly mean after rain and temperature, and the share of sites declining."""
    remainders = weather_remainders(table[table["year"].between(*period)], value)
    trend, low, high = trends.mean_trend(remainders)
    tested = trends.site_trends(remainders, min_years=min_years)
    return {"series": name, "trend per decade": round(trend, 3), "95% range": f"{low:+.3f} to {high:+.3f}",
            "sites declining": round((tested["p_decline"] < 0.05).mean(), 3)}

modis = pd.read_csv(paths.PROCESSED_DIR / "modis_jul_aug_median.csv")[["gid", "year", "ndvi"]]
landsat = pd.read_csv(paths.PROCESSED_DIR / "landsat_site_years.csv")[["gid", "year", "ndvi"]]
long_years = int(np.ceil(LONG_MIN_YEARS * 36))
series = [("biomass, 2011-2020", biomass, "bms_floored", MAIN_YEARS, MIN_YEARS),
          ("MODIS, 2011-2020", modis, "ndvi", MAIN_YEARS, MIN_YEARS),
          ("Landsat, 2011-2020", landsat, "ndvi", MAIN_YEARS, MIN_YEARS),
          ("MODIS, 2000-2024", modis, "ndvi", (2000, 2024), 20),
          ("Landsat, 1989-2024", landsat, "ndvi", (1989, 2024), long_years)]
compare = {"soum weather": pd.DataFrame([trend_row(name, with_soum_weather(table), value, period, least)
                                         for name, table, value, period, least in series]).set_index("series"),
           "site weather": pd.DataFrame([trend_row(name, with_site_weather(table), value, period, least)
                                         for name, table, value, period, least in series]).set_index("series")}
pd.concat(compare, axis=1)

soum weather                                    \
                   trend per decade         95% range sites declining   
series                                                                  
biomass, 2011-2020           -0.219  -0.403 to -0.034           0.205   
MODIS, 2011-2020             -0.007  -0.057 to +0.043           0.050   
Landsat, 2011-2020           -0.020  -0.088 to +0.048           0.080   
MODIS, 2000-2024              0.033  +0.020 to +0.045           0.005   
Landsat, 1989-2024            0.015  +0.004 to +0.027           0.036   

                       site weather                                    
                   trend per decade         95% range sites declining  
series                                                                 
biomass, 2011-2020           -0.212  -0.401 to -0.024           0.191  
MODIS, 2011-2020             -0.005  -0.057 to +0.048           0.054  
Landsat, 2011-2020           -0.017  -0.088 to +0.054           0.073  
MODIS, 2000-2024              0.033  +0.021 to +0.046           0.004  
Landsat, 1989-2024            0.015  +0.004 to +0.027           0.036

In [6]:
# Zone by zone, 2011-2020, with each site's own weather
zone_rows = []
for zone in ZONES:
    row = {"zone": zone}
    for name, table, value in [("biomass", biomass, "bms_floored"), ("MODIS", modis, "ndvi"), ("Landsat", landsat, "ndvi")]:
        weathered = with_site_weather(table)
        remainders = weather_remainders(weathered[weathered["year"].between(*MAIN_YEARS)], value)
        trend, low, high = trends.mean_trend(remainders[remainders["zone"] == zone])
        row[name] = round(trend, 3)
        row[name + " range"] = f"{low:+.3f} to {high:+.3f}"
    zone_rows.append(row)
pd.DataFrame(zone_rows)

,zone,biomass,biomass range,MODIS,MODIS range,Landsat,Landsat range
0,Desert zone,-0.401,-0.746 to -0.056,-0.033,-0.108 to +0.042,-0.055,-0.127 to +0.018
1,Semi desert steppe zone,-0.418,-0.666 to -0.169,-0.027,-0.077 to +0.024,-0.066,-0.127 to -0.004
2,Steppe zone,-0.232,-0.413 to -0.050,-0.009,-0.069 to +0.051,-0.016,-0.089 to +0.057
3,Forest steppe zone,0.017,-0.300 to +0.334,0.028,-0.052 to +0.109,0.031,-0.080 to +0.143
4,Mountain taiga belt,-0.023,-0.159 to +0.112,0.008,-0.037 to +0.054,0.014,-0.053 to +0.081


- **A site's own weather explains its biomass no better.** The share of year-to-year change that the weather explains is the same to within 0.005 in every zone.
- **No trend moves.** Biomass over 2011–2020 goes from −0.219 to −0.212, Landsat from −0.020 to −0.017 and MODIS from −0.007 to −0.005. The long records are the same to three decimals.
- **The zones keep their pattern.** Biomass declines in the three dry zones, and Landsat a little in the two driest.

## In short

- **The soum's weather was a sound stand-in** for the weather at the sites.
- **A limit.** ERA5-Land's grid cells are about 11 km across and come from a weather model. A storm that wets one valley and not the next is beyond it, and station records would be the stricter test.